In [ ]:
import os
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By

from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC




# 配置 Chrome 浏览器选项
options = Options()
options.add_argument(r"--user-data-dir=C:\Users\qieziclub1660ti\AppData\Local\Google\Chrome\SeleniumData")  # 替换为实际用户数据目录
options.add_argument("--start-maximized")  # 最大化窗口

# 创建 Chrome WebDriver
webdriver_path = r"C:\WebDriver\chromedriver.exe"

driver = webdriver.Chrome(
    service=Service(webdriver_path),
    options=options
)



# =========================
# 打开抖音创作者中心
# =========================

url = "https://creator.douyin.com/creator-micro/content/manage"

driver.get(url)


wait = WebDriverWait(driver,300)



# =========================
# 等待登录
# =========================

try:

    print("请扫码登录抖音创作者中心...")


    wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//*[contains(text(),'全部作品')]"
            )
        )
    )


    print(
        "登录成功，进入作品管理页面"
    )


except Exception as e:

    print(
        "登录失败:",
        e
    )

    driver.quit()
    exit()



# =========================
# 数字转换
# =========================

def convert_number(text):

    if not text:
        return 0


    text = text.strip()


    if "万" in text:

        try:

            return int(
                float(
                    text.replace(
                        "万",
                        ""
                    )
                )
                *
                10000
            )

        except:

            return 0


    try:

        return int(
            text.replace(",","")
        )

    except:

        return 0



# =========================
# 提取变量
# =========================

all_data = []



loaded_titles = set()



last_count = 0

no_change = 0



# =========================
# 滚动加载全部作品
# =========================

while True:


    # 找作品卡片

    cards = driver.find_elements(
        By.CSS_SELECTOR,
        "div[class^='video-card-info']"
    )


    print(
        f"当前发现作品:{len(cards)}"
    )



    for card in cards:


        try:


            # 标题

            title = card.find_element(
                By.CSS_SELECTOR,
                "div[class^='info-title-text']"
            ).text.strip()



            if not title:
                continue



            # 防止重复

            if title in loaded_titles:

                continue


            loaded_titles.add(title)



            # 时间

            try:

                date = card.find_element(
                    By.CSS_SELECTOR,
                    "div[class^='info-time']"
                ).text.strip()

            except:

                date = ""



            # 默认值

            play = 0

            like = 0

            comment = 0



            # 数据区域

            metrics = card.find_elements(
                By.CSS_SELECTOR,
                "div[class^='metric-item-container']"
            )



            for metric in metrics:


                try:


                    label = metric.find_element(
                        By.CSS_SELECTOR,
                        "div[class^='metric-label']"
                    ).text.strip()



                    value = metric.find_element(
                        By.CSS_SELECTOR,
                        "div[class^='metric-value']"
                    ).text.strip()



                    if label == "播放":

                        play = convert_number(
                            value
                        )


                    elif label == "点赞":

                        like = convert_number(
                            value
                        )


                    elif label == "评论":

                        comment = convert_number(
                            value
                        )


                except:

                    continue




            all_data.append(
                {
                    "标题":title,
                    "发布时间":date,
                    "播放量":play,
                    "点赞量":like,
                    "评论量":comment
                }
            )



            print(
                f"{title} | 播放:{play} | 点赞:{like} | 评论:{comment}"
            )



        except Exception as e:

            print(
                "作品提取失败:",
                e
            )



    # =========================
    # 判断是否加载完成
    # =========================


    current_count = len(cards)



    if current_count == last_count:

        no_change += 1

    else:

        no_change = 0



    last_count = current_count



    if no_change >= 3:

        print(
            "已经加载全部作品"
        )

        break



    # 滚动到底部

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight);"
    )


    print(
        "正在滚动加载..."
    )


    time.sleep(1)



# =========================
# 保存Excel
# =========================

print(
    f"最终作品数量:{len(all_data)}"
)



df = pd.DataFrame(
    all_data
)



file_path = os.path.join(
    os.path.expanduser("~"),
    "Desktop",
    "抖音作品数据.xlsx"
)



df.to_excel(
    file_path,
    index=False
)



print(
    f"保存完成:{file_path}"
)



driver.quit()



# 自动打开

os.startfile(
    file_path
)

请扫码登录抖音创作者中心...
登录成功，进入作品管理页面
当前发现作品:12
OCS光交换机需求爆了！MEMS微镜凭什么成为谷歌最优解？ #谷歌 #OCS #光子芯片 #MEMS #AI | 播放:20000 | 点赞:154 | 评论:8
半导体气体产业链创新成果路演圆满落幕！ 近距离感受产业一线的供需碰撞 #电子气体 #芯片 #半导体 #AI | 播放:2786 | 点赞:19 | 评论:1
对话Luceda：硅光大爆发，如何与广立微携手抢占产业新机遇 #CPO #硅光 #算力光互联 #光子芯片 从追赶到量产，硅光元年见证国产突破。本次对话嘉宾Luceda 总裁曹如平博士深耕集成光子赛道、见证硅光从科研走向商用。听她拆解光子 EDA 与传统芯片工具核心区别，解读硅光替代铜缆、支撑 CPO 演进完整路线，剖析产业链协同瓶颈，让我们一同站在算力光互联新时代。 | 播放:3037 | 点赞:30 | 评论:2
出席IPO敲钟仪式！幻实见证9年老友从一颗胎压芯片走到上市 #琻捷电子 #芯片 #半导体 #IPO | 播放:2073 | 点赞:15 | 评论:3
零失效+高交付！晶能领跑车规碳化硅主驱赛道 成立仅四年，主驱#碳化硅 模块 细分领域出货量全球第一，上车至今#零失效 。这一次，他们首次在第三方平台发声，把“向死而生”的故事带到了#幻实会客厅 。 本期对话嘉宾是一位跨商业航天、功率半导体、AI三大硬核科技领域的连续科技创新创业者——浙江晶能微电子有限公司创始人兼CEO潘运滨。听他讲述如何用“向死而生”的思考方式，为国产碳化硅打开局面，带领晶能主驱碳化硅模块市场中杀出重围，完成一次“小登顶”。 | 播放:4586 | 点赞:44 | 评论:3
73年老牌企业不只有设计：揭秘中国电子院的AI进化之路！ #设计院 #集成电路 #芯片 | 播放:304 | 点赞:5 | 评论:2
揭秘,打卡国内首条光子芯片中试线! 实地打卡光子芯片产线，从技术研发到产业孵化全流程解读，国产硬核科技未来可期！#光子芯片 #前沿科技 #芯片研发 | 播放:18000 | 点赞:275 | 评论:19
跳出单点局限，广立微解锁AI+EDA新战力！ #广立微 #AI #EDA #半导体 | 播放:2665 | 点赞:36 | 评论:5
幻实带你走进#无锡市集成电路学会 ，一探究竟#芯片 #集成